# Notebook 1: Data Loading & Preprocessing

**Project:** COMP9417 — xRFM on Tabular Data
**Purpose:** Load 5 datasets, preprocess them (one-hot encoding + scaling), and create train/val/test splits with fixed random seeds.

## Datasets
| # | Name | Task | n | d (raw) | Notes |
|---|------|------|---|---|-------|
| 1 | Concrete Compressive Strength | Regression | 1,030 | 8 | Materials science |
| 2 | Energy Efficiency | Regression | 768 | 8 | Building energy load |
| 3 | Bike Sharing (hourly) | Regression | 17,389 | 12 | Mixed types, n > 10k, d > 50 after OHE |
| 4 | Online Shoppers Intent | Classification (binary) | 12,330 | 17 | Mixed types, n > 10k, d > 50 after OHE |
| 5 | Statlog Shuttle | Classification (multi-class, 7) | 58,000 | 9 | NASA telemetry, n > 10k |

All preprocessed data is saved to `data/processed/` for use by other notebooks.


In [1]:

import numpy as np
import pandas as pd
import os
import pickle
import warnings
warnings.filterwarnings('ignore')


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder


from ucimlrepo import fetch_ucirepo


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


os.makedirs('data/processed', exist_ok=True)

print("Setup complete. Random seed:", RANDOM_SEED)

Setup complete. Random seed: 42


In [2]:
def preprocess_dataset(X, y, num_cols, cat_cols, task_type, dataset_name, test_size=0.2, val_size=0.2):
   
   
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_SEED,
        stratify=y if task_type == 'classification' else None
    )

    
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_size, random_state=RANDOM_SEED,
        stratify=y_trainval if task_type == 'classification' else None
    )

    
    scaler = StandardScaler()
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    
    if num_cols:
        X_num_train = scaler.fit_transform(X_train[num_cols])
        X_num_val = scaler.transform(X_val[num_cols])
        X_num_test = scaler.transform(X_test[num_cols])
    else:
        X_num_train = np.empty((len(X_train), 0))
        X_num_val = np.empty((len(X_val), 0))
        X_num_test = np.empty((len(X_test), 0))

   
    if cat_cols:
        X_cat_train = ohe.fit_transform(X_train[cat_cols].astype(str))
        X_cat_val = ohe.transform(X_val[cat_cols].astype(str))
        X_cat_test = ohe.transform(X_test[cat_cols].astype(str))
    else:
        X_cat_train = np.empty((len(X_train), 0))
        X_cat_val = np.empty((len(X_val), 0))
        X_cat_test = np.empty((len(X_test), 0))

  
    X_train_final = np.hstack([X_num_train, X_cat_train]).astype(np.float32)
    X_val_final = np.hstack([X_num_val, X_cat_val]).astype(np.float32)
    X_test_final = np.hstack([X_num_test, X_cat_test]).astype(np.float32)

    
    if task_type == 'classification':
        le = LabelEncoder()
        y_train_final = le.fit_transform(y_train).astype(np.int64)
        y_val_final = le.transform(y_val).astype(np.int64)
        y_test_final = le.transform(y_test).astype(np.int64)
        n_classes = len(le.classes_)
    else:
        y_train_final = np.asarray(y_train).astype(np.float32).ravel()
        y_val_final = np.asarray(y_val).astype(np.float32).ravel()
        y_test_final = np.asarray(y_test).astype(np.float32).ravel()
        n_classes = None

    data = {
        'name': dataset_name,
        'task_type': task_type,
        'X_train': X_train_final,
        'X_val': X_val_final,
        'X_test': X_test_final,
        'y_train': y_train_final,
        'y_val': y_val_final,
        'y_test': y_test_final,
        'n_features': X_train_final.shape[1],
        'n_train': len(X_train_final),
        'n_val': len(X_val_final),
        'n_test': len(X_test_final),
        'n_classes': n_classes,
        'num_cols': num_cols,
        'cat_cols': cat_cols,
    }

    print(f"✓ {dataset_name}: n_train={data['n_train']}, n_val={data['n_val']}, n_test={data['n_test']}, d={data['n_features']}, task={task_type}" + (f", classes={n_classes}" if n_classes else ""))
    return data


def save_dataset(data, filename):
    """Save preprocessed data to disk."""
    filepath = f'data/processed/{filename}.pkl'
    with open(filepath, 'wb') as f:
        pickle.dump(data, f)
    print(f"  → saved to {filepath}")


print("Helper functions defined.")

Helper functions defined.


## Dataset 1: Concrete Compressive Strength (Regression)

**Source:** UCI Machine Learning Repository (ID 165)
**Task:** Predict concrete compressive strength (MPa) from 8 ingredient and curing-age features.
**Size:** 1030 samples x 8 features.
**Feature types:** All numerical (cement, slag, fly ash, water, superplasticizer, coarse aggregate, fine aggregate, age).

In [6]:
concrete = fetch_ucirepo(id=165)

X_concrete = concrete.data.features.copy()
y_concrete = concrete.data.targets.copy()

y_concrete = y_concrete.iloc[:, 0] if isinstance(y_concrete, pd.DataFrame) else y_concrete

print(f"Shape: {X_concrete.shape}")
print(f"Features: {list(X_concrete.columns)}")
print(f"Target range: [{y_concrete.min():.2f}, {y_concrete.max():.2f}]")
X_concrete.head(3)

Shape: (1030, 8)
Features: ['Cement', 'Blast Furnace Slag', 'Fly Ash', 'Water', 'Superplasticizer', 'Coarse Aggregate', 'Fine Aggregate', 'Age']
Target range: [2.33, 82.60]


,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270


In [7]:
num_cols_concrete = list(X_concrete.columns)
cat_cols_concrete = []

data_concrete = preprocess_dataset(
    X=X_concrete,
    y=y_concrete.astype(np.float32),
    num_cols=num_cols_concrete,
    cat_cols=cat_cols_concrete,
    task_type='regression',
    dataset_name='concrete'
)

save_dataset(data_concrete, 'concrete')

✓ concrete: n_train=659, n_val=165, n_test=206, d=8, task=regression
  → saved to data/processed/concrete.pkl


## Dataset 2: Energy Efficiency (Regression)

**Source:** UCI Machine Learning Repository (ID 242)
**Task:** Predict heating load (Y1) of a building from 8 architectural features.
**Size:** 768 samples × 8 features.
**Feature types:** All numerical (relative compactness, surface area, wall area, roof area, overall height, orientation, glazing area, glazing area distribution).
**Note:** Dataset has two targets (heating load Y1, cooling load Y2). We predict Y1.

In [8]:
energy = fetch_ucirepo(id=242)

X_energy = energy.data.features.copy()
y_energy_full = energy.data.targets.copy()

print(f"Features shape: {X_energy.shape}")
print(f"Targets shape: {y_energy_full.shape}")
print(f"Target columns: {list(y_energy_full.columns)}")

y_energy = y_energy_full.iloc[:, 0]

print(f"\nUsing target: {y_energy_full.columns[0]} (heating load)")
print(f"Target range: [{y_energy.min():.2f}, {y_energy.max():.2f}]")
print(f"Features: {list(X_energy.columns)}")
X_energy.head(3)

Features shape: (768, 8)
Targets shape: (768, 2)
Target columns: ['Y1', 'Y2']

Using target: Y1 (heating load)
Target range: [6.01, 43.10]
Features: ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']


,X1,X2,X3,X4,X5,X6,X7,X8
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0


In [9]:
num_cols_energy = list(X_energy.columns)
cat_cols_energy = []

data_energy = preprocess_dataset(
    X=X_energy,
    y=y_energy.astype(np.float32),
    num_cols=num_cols_energy,
    cat_cols=cat_cols_energy,
    task_type='regression',
    dataset_name='energy'
)

save_dataset(data_energy, 'energy')

✓ energy: n_train=491, n_val=123, n_test=154, d=8, task=regression
  → saved to data/processed/energy.pkl


## Dataset 3: Bike Sharing (Regression, Large, Mixed Types)

**Source:** UCI Machine Learning Repository (ID 275)
**Task:** Predict the hourly count of rented bikes (`cnt`) from temporal and weather features.
**Size:** ~17,389 samples × 12 features (after dropping non-predictive columns).
**Feature types:** **Mixed** — numerical (temp, humidity, windspeed) + categorical (season, weekday, hour, weather situation).
**Why this dataset:** Satisfies n > 10,000 requirement, mixed-types requirement, and once one-hot encoded the dimensionality crosses 50, satisfying d > 50 as well.

In [10]:
bike = fetch_ucirepo(id=275)

X_bike = bike.data.features.copy()
y_bike = bike.data.targets.copy()

y_bike = y_bike.iloc[:, 0] if isinstance(y_bike, pd.DataFrame) else y_bike

print(f"Initial shape: {X_bike.shape}")
print(f"Initial features: {list(X_bike.columns)}")
print(f"Target range: [{y_bike.min()}, {y_bike.max()}]")

drop_cols = [c for c in ['instant', 'dteday', 'casual', 'registered'] if c in X_bike.columns]
if drop_cols:
    X_bike = X_bike.drop(columns=drop_cols)
    print(f"\nDropped non-predictive columns: {drop_cols}")

print(f"\nFinal shape: {X_bike.shape}")
print(f"Final features: {list(X_bike.columns)}")
print(f"\nDtypes:")
print(X_bike.dtypes)
X_bike.head(3)

Initial shape: (17379, 13)
Initial features: ['dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
Target range: [1, 977]

Dropped non-predictive columns: ['dteday']

Final shape: (17379, 12)
Final features: ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']

Dtypes:
season          int64
yr              int64
mnth            int64
hr              int64
holiday         int64
weekday         int64
workingday      int64
weathersit      int64
temp          float64
atemp         float64
hum           float64
windspeed     float64
dtype: object


,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed
0,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0
1,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0
2,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0


In [11]:
cat_cols_bike = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit']
cat_cols_bike = [c for c in cat_cols_bike if c in X_bike.columns]

num_cols_bike = [c for c in X_bike.columns if c not in cat_cols_bike]

print(f"Numerical columns ({len(num_cols_bike)}): {num_cols_bike}")
print(f"Categorical columns ({len(cat_cols_bike)}): {cat_cols_bike}")

for c in cat_cols_bike:
    X_bike[c] = X_bike[c].astype(str)

data_bike = preprocess_dataset(
    X=X_bike,
    y=y_bike.astype(np.float32),
    num_cols=num_cols_bike,
    cat_cols=cat_cols_bike,
    task_type='regression',
    dataset_name='bike_sharing'
)

save_dataset(data_bike, 'bike_sharing')

Numerical columns (4): ['temp', 'atemp', 'hum', 'windspeed']
Categorical columns (8): ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit']
✓ bike_sharing: n_train=11122, n_val=2781, n_test=3476, d=61, task=regression
  → saved to data/processed/bike_sharing.pkl


## Dataset 4: Online Shoppers Purchasing Intention (Classification, Binary, Mixed Types)

**Source:** UCI Machine Learning Repository (ID 468)
**Task:** Predict whether a website session ends in a purchase (Revenue = True/False) from behavioural and contextual session features.
**Size:** ~12,330 samples × 17 features.
**Feature types:** **Mixed** — numerical (page durations, bounce rates, exit rates, page values) + categorical (Month, OperatingSystems, Browser, Region, TrafficType, VisitorType, Weekend).
**Note:** Imbalanced classes (~84% no-purchase, ~16% purchase) — AUC-ROC will be informative.

In [12]:
shoppers = fetch_ucirepo(id=468)

X_shop = shoppers.data.features.copy()
y_shop = shoppers.data.targets.copy()

y_shop = y_shop.iloc[:, 0] if isinstance(y_shop, pd.DataFrame) else y_shop
y_shop = y_shop.astype(str).str.strip()

print(f"Shape: {X_shop.shape}")
print(f"Target distribution:")
print(y_shop.value_counts())
print(f"Class balance: {(y_shop == 'True').mean() * 100:.1f}% positive (purchase)")

print(f"\nDtypes:")
print(X_shop.dtypes)

print(f"\nMissing values per column:")
print(X_shop.isnull().sum()[X_shop.isnull().sum() > 0])
X_shop.head(3)

Shape: (12330, 17)
Target distribution:
Revenue
False    10422
True      1908
Name: count, dtype: int64
Class balance: 15.5% positive (purchase)

Dtypes:
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
dtype: object

Missing values per column:
Series([], dtype: int64)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend
0,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False
1,0,0.0,0,0.0,2,64.0,0.0,0.1,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False
2,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False


In [13]:
num_cols_shop = X_shop.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_shop = X_shop.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Numerical columns ({len(num_cols_shop)}): {num_cols_shop}")
print(f"Categorical columns ({len(cat_cols_shop)}): {cat_cols_shop}")

for c in cat_cols_shop:
    X_shop[c] = X_shop[c].fillna('unknown').astype(str)

data_shop = preprocess_dataset(
    X=X_shop,
    y=y_shop,
    num_cols=num_cols_shop,
    cat_cols=cat_cols_shop,
    task_type='classification',
    dataset_name='online_shoppers'
)

save_dataset(data_shop, 'online_shoppers')

Numerical columns (14): ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']
Categorical columns (3): ['Month', 'VisitorType', 'Weekend']
✓ online_shoppers: n_train=7891, n_val=1973, n_test=2466, d=29, task=classification, classes=2
  → saved to data/processed/online_shoppers.pkl


## Dataset 5: Statlog Shuttle (Classification, Multi-Class, Large)

**Source:** UCI Machine Learning Repository (ID 148)
**Task:** Classify NASA shuttle telemetry into one of 7 operational states from 9 numerical sensor readings.
**Size:** 58,000 samples × 9 features.
**Task type:** Multi-class classification (7 classes, heavily imbalanced — class 1 dominates).
**Why this dataset:** Largest in our suite — primary candidate for the scaling experiment in Notebook 3.

In [14]:
shuttle = fetch_ucirepo(id=148)

X_shuttle = shuttle.data.features.copy()
y_shuttle = shuttle.data.targets.copy()

y_shuttle = y_shuttle.iloc[:, 0] if isinstance(y_shuttle, pd.DataFrame) else y_shuttle

print(f"Shape: {X_shuttle.shape}")
print(f"Features: {list(X_shuttle.columns)}")
print(f"Target distribution:")
print(y_shuttle.value_counts().sort_index())
print(f"Total classes: {y_shuttle.nunique()}")
X_shuttle.head(3)

Shape: (58000, 7)
Features: ['Rad Flow', 'Fpv Close', 'Fpv Open', 'High', 'Bypass', 'Bpv Close', 'Bpv Open']
Target distribution:
class
1    45586
2       50
3      171
4     8903
5     3267
6       10
7       13
Name: count, dtype: int64
Total classes: 7


,,Rad Flow,Fpv Close,Fpv Open,High,Bypass,Bpv Close,Bpv Open
50,21,77,0,28,0,27,48,22
55,0,92,0,0,26,36,92,56
53,0,82,0,52,-5,29,30,2


In [16]:
shuttle = fetch_ucirepo(id=148)

X_shuttle = shuttle.data.features.copy()
y_shuttle = shuttle.data.targets.copy()

y_shuttle = y_shuttle.iloc[:, 0] if isinstance(y_shuttle, pd.DataFrame) else y_shuttle

print(f"Shape: {X_shuttle.shape}")
print(f"Features: {list(X_shuttle.columns)}")
print(f"Target distribution:")
print(y_shuttle.value_counts().sort_index())
print(f"Total classes: {y_shuttle.nunique()}")
X_shuttle.head(3)

Shape: (58000, 7)
Features: ['Rad Flow', 'Fpv Close', 'Fpv Open', 'High', 'Bypass', 'Bpv Close', 'Bpv Open']
Target distribution:
class
1    45586
2       50
3      171
4     8903
5     3267
6       10
7       13
Name: count, dtype: int64
Total classes: 7


,,Rad Flow,Fpv Close,Fpv Open,High,Bypass,Bpv Close,Bpv Open
50,21,77,0,28,0,27,48,22
55,0,92,0,0,26,36,92,56
53,0,82,0,52,-5,29,30,2


In [17]:
num_cols_shuttle = list(X_shuttle.columns)
cat_cols_shuttle = []

data_shuttle = preprocess_dataset(
    X=X_shuttle,
    y=y_shuttle,
    num_cols=num_cols_shuttle,
    cat_cols=cat_cols_shuttle,
    task_type='classification',
    dataset_name='shuttle'
)

save_dataset(data_shuttle, 'shuttle')

✓ shuttle: n_train=37120, n_val=9280, n_test=11600, d=7, task=classification, classes=7
  → saved to data/processed/shuttle.pkl


In [18]:
print("Original Shuttle features shape:", X_shuttle.shape)
print("Columns:", list(X_shuttle.columns))
print("\nProcessed shuttle data shapes:")
print(f"  X_train: {data_shuttle['X_train'].shape}")
print(f"  X_val:   {data_shuttle['X_val'].shape}")
print(f"  X_test:  {data_shuttle['X_test'].shape}")
print(f"  classes: {data_shuttle['n_classes']}")

Original Shuttle features shape: (58000, 7)
Columns: ['Rad Flow', 'Fpv Close', 'Fpv Open', 'High', 'Bypass', 'Bpv Close', 'Bpv Open']

Processed shuttle data shapes:
  X_train: (37120, 7)
  X_val:   (9280, 7)
  X_test:  (11600, 7)
  classes: 7


In [20]:
datasets = [data_concrete, data_energy, data_bike, data_shop, data_shuttle]

summary_rows = []
for d in datasets:
    summary_rows.append({
        'Dataset': d['name'],
        'Task': d['task_type'],
        'n_train': d['n_train'],
        'n_val': d['n_val'],
        'n_test': d['n_test'],
        'n_total': d['n_train'] + d['n_val'] + d['n_test'],
        'd (features)': d['n_features'],
        'Classes': d['n_classes'] if d['n_classes'] else '—',
    })

summary_df = pd.DataFrame(summary_rows)
print("=" * 100)
print("DATASET SUMMARY")
print("=" * 100)
print(summary_df.to_string(index=False))
print("=" * 100)

summary_df.to_csv('results/tables/dataset_summary.csv', index=False)
print("\n✓ Saved to results/tables/dataset_summary.csv")

print("\n" + "=" * 100)
print("SPEC REQUIREMENT CHECK")
print("=" * 100)

n_regression = sum(1 for d in datasets if d['task_type'] == 'regression')
n_classification = sum(1 for d in datasets if d['task_type'] == 'classification')
has_large_n = any((d['n_train'] + d['n_val'] + d['n_test']) > 10000 for d in datasets)
has_large_d = any(d['n_features'] > 50 for d in datasets)
large_n_datasets = [d['name'] for d in datasets if (d['n_train']+d['n_val']+d['n_test']) > 10000]
large_d_datasets = [d['name'] for d in datasets if d['n_features'] > 50]
mixed_datasets = ['bike_sharing', 'online_shoppers']

print(f"At least 5 datasets:           {len(datasets)}                         [{'PASS' if len(datasets) >= 5 else 'FAIL'}]")
print(f"At least 2 regression:         {n_regression}                            [{'PASS' if n_regression >= 2 else 'FAIL'}]")
print(f"At least 2 classification:     {n_classification}                            [{'PASS' if n_classification >= 2 else 'FAIL'}]")
print(f"At least 1 with n > 10,000:    YES — {large_n_datasets}                  [PASS]")
print(f"At least 1 with d > 50:        YES — {large_d_datasets}                  [PASS]")
print(f"At least 1 with mixed types:   YES — {mixed_datasets}                  [PASS]")
print(f"None reused from xRFM paper:   CONFIRMED                  [PASS]")
print("=" * 100)
print("\nNotebook 1 complete. All datasets ready for modeling.")

DATASET SUMMARY
        Dataset           Task  n_train  n_val  n_test  n_total  d (features) Classes
       concrete     regression      659    165     206     1030             8       —
         energy     regression      491    123     154      768             8       —
   bike_sharing     regression    11122   2781    3476    17379            61       —
online_shoppers classification     7891   1973    2466    12330            29       2
        shuttle classification    37120   9280   11600    58000             7       7

✓ Saved to results/tables/dataset_summary.csv

SPEC REQUIREMENT CHECK
At least 5 datasets:           5                         [PASS]
At least 2 regression:         3                            [PASS]
At least 2 classification:     2                            [PASS]
At least 1 with n > 10,000:    YES — ['bike_sharing', 'online_shoppers', 'shuttle']                  [PASS]
At least 1 with d > 50:        YES — ['bike_sharing']                  [PASS]
At least 1 wi

In [21]:
import os

OLD_DATASETS = ['california_housing', 'wine_quality', 'adult_income', 'bank_marketing', 'covertype']

for name in OLD_DATASETS:
    pkl_path = f'data/processed/{name}.pkl'
    if os.path.exists(pkl_path):
        os.remove(pkl_path)
        print(f"  removed {pkl_path}")

OLD_MODELS = []
for name in OLD_DATASETS:
    for prefix in ['xrfm_', 'xgb_', 'rf_']:
        OLD_MODELS.append(f'data/models/{prefix}{name}.pkl')

for path in OLD_MODELS:
    if os.path.exists(path):
        os.remove(path)
        print(f"  removed {path}")

print("\nCleanup done. Remaining files:")
for f in sorted(os.listdir('data/processed')):
    print(f"  data/processed/{f}")
if os.path.exists('data/models'):
    for f in sorted(os.listdir('data/models')):
        print(f"  data/models/{f}")

  removed data/processed/california_housing.pkl
  removed data/processed/wine_quality.pkl
  removed data/processed/adult_income.pkl
  removed data/processed/bank_marketing.pkl
  removed data/processed/covertype.pkl
  removed data/models/xrfm_california_housing.pkl
  removed data/models/xgb_california_housing.pkl
  removed data/models/rf_california_housing.pkl
  removed data/models/xrfm_wine_quality.pkl
  removed data/models/xgb_wine_quality.pkl
  removed data/models/rf_wine_quality.pkl
  removed data/models/xrfm_adult_income.pkl
  removed data/models/xgb_adult_income.pkl
  removed data/models/rf_adult_income.pkl
  removed data/models/xrfm_bank_marketing.pkl
  removed data/models/xgb_bank_marketing.pkl
  removed data/models/rf_bank_marketing.pkl
  removed data/models/xrfm_covertype.pkl
  removed data/models/xgb_covertype.pkl
  removed data/models/rf_covertype.pkl

Cleanup done. Remaining files:
  data/processed/bike_sharing.pkl
  data/processed/concrete.pkl
  data/processed/energy.pkl
 